# RNCP C3 — Data Dictionary and Multi-source Data Integration

## Step 1 — Data Source Inspection

This workflow integrates metadata from three credit card-related data sources:

- **UCI Dataset:** Historical customer-level credit card default data from Taiwan.
- **NCCC API:** Annual credit card cash advance statistics.
- **FSC Web Scraping:** Public credit card market indicators.

The objective is to document data sources, assess data quality, standardize metadata, and create a reproducible integration workflow.

Because the datasets have different time periods and levels of aggregation, they are treated as complementary sources rather than directly joined customer-level records.

In [1]:

import pandas as pd

# Load the three existing CSV files
uci_df = pd.read_csv("credit_card_default_clean.csv")
api_df = pd.read_csv("credit_card_api.csv")
scraping_df = pd.read_csv("credit_card_scraping.csv")

# Organize data sources
data_sources = {
    "UCI Dataset": uci_df,
    "NCCC API": api_df,
    "FSC Web Scraping": scraping_df
}

# Inspect dataset structure and missing values
summary = []

for name, data in data_sources.items():
    summary.append({
        "data_source": name,
        "rows": len(data),
        "columns": len(data.columns),
        "missing_values": int(data.isna().sum().sum()),
        "duplicate_rows": int(data.duplicated().sum())
    })

source_summary = pd.DataFrame(summary)

display(source_summary)

# Display the column names
for name, data in data_sources.items():
    print(f"\n{name}:")
    print(data.columns.tolist())


,data_source,rows,columns,missing_values,duplicate_rows
0,UCI Dataset,30000,25,0,0
1,NCCC API,17,4,0,0
2,FSC Web Scraping,2,4,0,0



UCI Dataset:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'default payment next month']

NCCC API:
['year', 'cash_advance_amount_100m_twd', 'cash_advance_transactions_1000', 'source']

FSC Web Scraping:
['indicator', 'value', 'source', 'source_url']


## Step 2 — Data Dictionary

A data dictionary is created to document the structure of the three data sources.

For each field, the dictionary records its source, name, data type, description, measurement unit, and data granularity.

The dictionary supports data traceability, consistency, and documentation across the data engineering workflow.

In [2]:

# Define source-level metadata
source_metadata = {
    "UCI Dataset": {
        "granularity": "Customer-level",
        "period": "2005"
    },
    "NCCC API": {
        "granularity": "Annual industry-level",
        "period": "2009-2025"
    },
    "FSC Web Scraping": {
        "granularity": "Industry indicator-level",
        "period": "As published on source website"
    }
}

# Define known field descriptions and units
field_details = {
    "ID": ("Customer identifier", "ID"),
    "LIMIT_BAL": ("Credit limit", "NTD"),
    "SEX": ("Encoded sex category", "Category code"),
    "EDUCATION": ("Encoded education category", "Category code"),
    "MARRIAGE": ("Encoded marital status", "Category code"),
    "AGE": ("Customer age", "Years"),
    "year": ("Reporting year", "Year"),
    "cash_advance_amount_100m_twd": (
        "Annual cash advance amount", "100 million TWD"
    ),
    "cash_advance_transactions_1000": (
        "Annual cash advance transactions", "1000 transactions"
    ),
    "indicator": ("Financial indicator name", "Text"),
    "value": ("Reported indicator value", "Indicator-specific"),
    "source": ("Data source organization", "Text"),
    "source_url": ("Original webpage URL", "URL")
}

# Generate data dictionary
dictionary_records = []

for source_name, data in data_sources.items():
    for column in data.columns:
        description, unit = field_details.get(
            column,
            ("See original dataset documentation", "Not specified")
        )

        dictionary_records.append({
            "source": source_name,
            "field_name": column,
            "data_type": str(data[column].dtype),
            "description": description,
            "unit": unit,
            "granularity": source_metadata[source_name]["granularity"],
            "period": source_metadata[source_name]["period"]
        })

data_dictionary = pd.DataFrame(dictionary_records)

display(data_dictionary)

# Export data dictionary
data_dictionary.to_csv(
    "data_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Data dictionary exported successfully!")
print("Total fields:", len(data_dictionary))


,source,field_name,data_type,description,unit,granularity,period
0,UCI Dataset,ID,int64,Customer identifier,ID,Customer-level,2005
1,UCI Dataset,LIMIT_BAL,int64,Credit limit,NTD,Customer-level,2005
2,UCI Dataset,SEX,int64,Encoded sex category,Category code,Customer-level,2005
3,UCI Dataset,EDUCATION,int64,Encoded education category,Category code,Customer-level,2005
4,UCI Dataset,MARRIAGE,int64,Encoded marital status,Category code,Customer-level,2005
5,UCI Dataset,AGE,int64,Customer age,Years,Customer-level,2005
6,UCI Dataset,PAY_0,int64,See original dataset documentation,Not specified,Customer-level,2005
7,UCI Dataset,PAY_2,int64,See original dataset documentation,Not specified,Customer-level,2005
8,UCI Dataset,PAY_3,int64,See original dataset documentation,Not specified,Customer-level,2005
9,UCI Dataset,PAY_4,int64,See original dataset documentation,Not specified,Customer-level,2005


Data dictionary exported successfully!
Total fields: 33


## Step 2.1 — Complete Field Descriptions

In [3]:

# Complete the UCI field descriptions
for field in data_dictionary["field_name"]:
    if field.startswith("PAY_") and field != "PAY_0" or field == "PAY_0":
        mask = (
            (data_dictionary["source"] == "UCI Dataset")
            & (data_dictionary["field_name"] == field)
        )
        data_dictionary.loc[mask, "description"] = (
            "Monthly repayment status (historical)"
        )
        data_dictionary.loc[mask, "unit"] = "Repayment status code"

    elif field.startswith("BILL_AMT"):
        mask = (
            (data_dictionary["source"] == "UCI Dataset")
            & (data_dictionary["field_name"] == field)
        )
        data_dictionary.loc[mask, "description"] = (
            "Monthly credit card bill amount"
        )
        data_dictionary.loc[mask, "unit"] = "NTD"

    elif field.startswith("PAY_AMT"):
        mask = (
            (data_dictionary["source"] == "UCI Dataset")
            & (data_dictionary["field_name"] == field)
        )
        data_dictionary.loc[mask, "description"] = (
            "Monthly previous payment amount"
        )
        data_dictionary.loc[mask, "unit"] = "NTD"

# Complete the target column description
target_mask = (
    (data_dictionary["source"] == "UCI Dataset")
    & data_dictionary["field_name"].str.contains(
        "default", case=False
    )
)

data_dictionary.loc[target_mask, "description"] = (
    "Default payment in the following month "
    "(0 = No Default, 1 = Default)"
)
data_dictionary.loc[target_mask, "unit"] = "Binary label"

# Save the updated dictionary
data_dictionary.to_csv(
    "data_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

display(data_dictionary.tail(10))

print("Data dictionary updated successfully!")


,source,field_name,data_type,description,unit,granularity,period
23,UCI Dataset,PAY_AMT6,int64,Monthly repayment status (historical),Repayment status code,Customer-level,2005
24,UCI Dataset,default payment next month,int64,Default payment in the following month (0 = No...,Binary label,Customer-level,2005
25,NCCC API,year,int64,Reporting year,Year,Annual industry-level,2009-2025
26,NCCC API,cash_advance_amount_100m_twd,int64,Annual cash advance amount,100 million TWD,Annual industry-level,2009-2025
27,NCCC API,cash_advance_transactions_1000,int64,Annual cash advance transactions,1000 transactions,Annual industry-level,2009-2025
28,NCCC API,source,object,Data source organization,Text,Annual industry-level,2009-2025
29,FSC Web Scraping,indicator,object,Financial indicator name,Text,Industry indicator-level,As published on source website
30,FSC Web Scraping,value,float64,Reported indicator value,Indicator-specific,Industry indicator-level,As published on source website
31,FSC Web Scraping,source,object,Data source organization,Text,Industry indicator-level,As published on source website
32,FSC Web Scraping,source_url,object,Original webpage URL,URL,Industry indicator-level,As published on source website


Data dictionary updated successfully!


## Step 3 — Data Standardization and Multi-source Integration

The three data sources have different structures, time periods, and levels of granularity.

The UCI customer dataset is retained separately for machine learning and customer-level analysis.

The NCCC API and FSC web scraping data are standardized into a common industry-indicator format, with consistent field names, units, source information, and reporting periods.

This integration preserves data provenance and avoids inappropriate joins between historical customer records and more recent industry statistics.

In [4]:

# Standardize NCCC API data
api_indicators = api_df.melt(
    id_vars=["year", "source"],
    value_vars=[
        "cash_advance_amount_100m_twd",
        "cash_advance_transactions_1000"
    ],
    var_name="indicator",
    value_name="value"
)

api_indicators["unit"] = api_indicators["indicator"].map({
    "cash_advance_amount_100m_twd": "100 million TWD",
    "cash_advance_transactions_1000": "1000 transactions"
})

api_indicators["reporting_period"] = (
    api_indicators["year"].astype(str)
)
api_indicators["granularity"] = "Annual industry-level"

# Standardize FSC web scraping data
fsc_indicators = scraping_df.copy()

fsc_indicators["unit"] = fsc_indicators["indicator"].map({
    "credit_cards_in_circulation": "1000 cards",
    "aggregate_credit_card_debt": "NTD billion"
})

fsc_indicators["reporting_period"] = "Not specified in extracted CSV"
fsc_indicators["granularity"] = "Industry indicator-level"

# Align columns
standard_columns = [
    "source",
    "indicator",
    "value",
    "unit",
    "reporting_period",
    "granularity"
]

# Combine industry-level records
industry_data = pd.concat(
    [
        api_indicators[standard_columns],
        fsc_indicators[standard_columns]
    ],
    ignore_index=True
)

# Validate standardized data
assert industry_data["unit"].notna().all()
assert industry_data["value"].notna().all()

display(industry_data.head())

# Export integrated industry data
industry_data.to_csv(
    "integrated_industry_data.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Integrated records:", len(industry_data))
print("CSV exported successfully!")


,source,indicator,value,unit,reporting_period,granularity
0,NCCC Taiwan,cash_advance_amount_100m_twd,40.0,100 million TWD,2009,Annual industry-level
1,NCCC Taiwan,cash_advance_amount_100m_twd,40.0,100 million TWD,2010,Annual industry-level
2,NCCC Taiwan,cash_advance_amount_100m_twd,38.0,100 million TWD,2011,Annual industry-level
3,NCCC Taiwan,cash_advance_amount_100m_twd,36.0,100 million TWD,2012,Annual industry-level
4,NCCC Taiwan,cash_advance_amount_100m_twd,34.0,100 million TWD,2013,Annual industry-level


Integrated records: 36
CSV exported successfully!


## Step 4 — Final Data Quality Validation

The integrated industry dataset is checked for missing values, duplicate records, and inconsistent data types.

The validation also verifies that each indicator has a defined measurement unit and that the expected number of records is preserved.

These checks support the reliability and reproducibility of the multi-source data integration workflow.

In [5]:

# Final data quality checks
quality_results = {
    "total_records": len(industry_data),
    "missing_values": int(industry_data.isna().sum().sum()),
    "duplicate_rows": int(industry_data.duplicated().sum()),
    "unique_sources": industry_data["source"].nunique(),
    "unique_indicators": industry_data["indicator"].nunique(),
    "missing_units": int(industry_data["unit"].isna().sum())
}

quality_df = pd.DataFrame(
    quality_results.items(),
    columns=["check", "result"]
)

display(quality_df)

# Validation assertions
assert len(industry_data) == 36
assert industry_data.isna().sum().sum() == 0
assert industry_data.duplicated().sum() == 0
assert industry_data["unit"].notna().all()

print("All data quality checks passed!")


,check,result
0,total_records,36
1,missing_values,0
2,duplicate_rows,0
3,unique_sources,2
4,unique_indicators,4
5,missing_units,0


All data quality checks passed!
